# Playground — Step 5: Family 1 Frame-Space Composition

Family 1 composes concepts **before** scoring, then runs vanilla single-concept CGD:

| method | operation | semantics |
|---|---|---|
| `Concept.average(concepts, weights)` | F1.a — weighted chordal (Procrustes) mean | compromise point between concepts |
| `Concept.joint_subspace(concepts)` | F1.b — SVD basis of the union of spans | OR: alignment with any constituent counts |

Quick wrappers: `quick_generate_with_topk_mean_frame_guide` (supports `guide_weights`,
negative weights repel) and `quick_generate_with_topk_subspace_guide` (now the *real*
subspace — before Step 5 that name incorrectly ran the mean).

Unequal-rank policy (documented in `Concept.average`): zero-pad to common k —
rank-neutral since rank counts nonzero vectors. `Concept.average` is the **extrinsic**
mean, not the geodesic Fréchet mean; the Riemannian variants (thesis RQ3) drop in there.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `frames` imports work

In [ ]:
import torch

from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 3
STEPS = 16


def chat(user: str, assistant: str = "") -> str:
    return (
        f"<|start_header_id|>user<|end_header_id|>{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{assistant}"
    )


def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]


PROMPTS = [
    chat("Tell me a short story.", "Once"),
    chat("Name things that make people happy.", "1."),
]

joy = fur.get_concept_cached("joy.n.01", MIN_LEMMAS, MAX_TOKENS)
dog = fur.get_concept_cached("dog.n.01", MIN_LEMMAS, MAX_TOKENS)

## 1 — Geometry first: where do the composites land?

ρ of each composite against its constituents. The mean is a compromise (roughly
equidistant); the subspace should correlate with **both** constituents better than
either correlates with the other.

In [ ]:
mean_c = Concept.average([joy, dog])
sub_c = Concept.joint_subspace([joy, dog])

print(f"{mean_c}")
print(f"{sub_c}   <- rank adds up (minus overlaps)")
print()
print(f"rho(joy, dog)       = {joy.rho(dog).item():+.3f}")
print(f"rho(mean, joy)      = {mean_c.rho(joy).item():+.3f}")
print(f"rho(mean, dog)      = {mean_c.rho(dog).item():+.3f}")
print(f"rho(subspace, joy)  = {sub_c.rho(joy).item():+.3f}")
print(f"rho(subspace, dog)  = {sub_c.rho(dog).item():+.3f}")

## 2 — Weighted mean: sweep the balance

`guide_weights` biases the compromise. Only ratios matter (the polar factor is
scale-invariant).

In [ ]:
for w_joy, w_dog in [(0.9, 0.1), (0.5, 0.5), (0.1, 0.9)]:
    texts, _ = fur.quick_generate_with_topk_mean_frame_guide(
        PROMPTS[:1],
        guides=[["joy.n.01"], ["dog.n.01"]],
        guide_weights=[w_joy, w_dog],
        min_lemmas_per_synset=MIN_LEMMAS,
        max_token_count=MAX_TOKENS,
        k=K,
        steps=STEPS,
    )
    print(f"joy={w_joy}, dog={w_dog}:", answer(texts[0])[:120])
    print()

## 3 — F1.a mean vs F1.b subspace in generation

In [ ]:
texts_mean, _ = fur.quick_generate_with_topk_mean_frame_guide(
    PROMPTS,
    guides=[["joy.n.01"], ["dog.n.01"]],
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=K,
    steps=STEPS,
)
texts_sub, _ = fur.quick_generate_with_topk_subspace_guide(
    PROMPTS,
    guides=[["joy.n.01"], ["dog.n.01"]],
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=K,
    steps=STEPS,
)

for mean_t, sub_t in zip(texts_mean, texts_sub):
    print("F1.a mean:    ", answer(mean_t)[:130])
    print("F1.b subspace:", answer(sub_t)[:130])
    print()

## 4 — Frame-space negative weights

`guide_weights=[1, -1]` in the mean is the frame-space analogue of score-space
repulsion — and closely related to the paper's differential guidance `joy - dog`
(which is Procrustes of the difference).

In [ ]:
neg_mean = Concept.average([joy, dog], weights=[1.0, -1.0])
diff = joy - dog
print(f"rho(weighted-mean[1,-1], joy - dog) = {neg_mean.rho(diff).item():+.4f}")

texts_neg, _ = fur.quick_generate_with_topk_mean_frame_guide(
    PROMPTS[:1],
    guides=[["joy.n.01"], ["dog.n.01"]],
    guide_weights=[1.0, -1.0],
    min_lemmas_per_synset=MIN_LEMMAS,
    max_token_count=MAX_TOKENS,
    k=K,
    steps=STEPS,
)
print("\njoy attract, dog repel:", answer(texts_neg[0])[:130])

## 5 — Composition revisited: does the subspace beat the mean at containing `girl`?

Step 3 found ρ(mean(woman, child), girl) = 0.49 > either constituent. The subspace
contains both constituents' spans entirely — compare.

In [ ]:
woman = fur.get_concept_cached("woman.n.01", MIN_LEMMAS, MAX_TOKENS)
child = fur.get_concept_cached("child.n.01", MIN_LEMMAS, MAX_TOKENS)
girl = fur.get_concept_cached("girl.n.01", MIN_LEMMAS, MAX_TOKENS)

mean_wc = Concept.average([woman, child])
sub_wc = Concept.joint_subspace([woman, child])

print(f"rho(woman, girl)          = {woman.rho(girl).item():+.3f}")
print(f"rho(child, girl)          = {child.rho(girl).item():+.3f}")
print(f"rho(mean(w,c), girl)      = {mean_wc.rho(girl).item():+.3f}")
print(f"rho(subspace(w,c), girl)  = {sub_wc.rho(girl).item():+.3f}")